# Water Quality data validation

For Lake Victoria

Compared against in-situ data downloaded from GEMStat data portal https://portal.gemstat.org/applications/public.html?publicuser=PublicUser#gemstat/Stations

The in-situ data have been filtered and extracted into .csv files

In [ ]:
%matplotlib inline

import datacube
import numpy as np
import matplotlib.pyplot as plt
#import matplotlib.ticker as mticker
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import seaborn as sns
from odc.geo.geom import Geometry
#from deafrica_tools.plotting import display_map
#from deafrica_tools.areaofinterest import define_area

In [ ]:
def flag_type(flag):
    if pd.isna(flag):
        return "normal"
    elif "<" in str(flag):
        return "below"
    elif ">" in str(flag):
        return "above"
    else:
        return "normal"

In [ ]:
def read_insitu(parameter):

    df = pd.read_csv(f"samples_{parameter}.csv")

    df["datetime"]  = pd.to_datetime(df['Sample Date'] + " " + df['Sample Time'], errors='coerce')
    df = df[df["Station Identifier"].str.contains("Lake Victoria")]
    df['flag_type'] = df['Value Flags'].apply(flag_type)

    units = df['Unit'].unique()
    if len(units)>1:
        to_update = df.Unit.str.startswith('mg')
        df.loc[to_update, 'Value'] = df.loc[to_update, 'Value']/1000.
        df.loc[to_update, 'Unit'] = df.loc[to_update, 'Unit'].str.replace(r'^m', 'µ', regex=True)
        units = df['Unit'].unique()
        if len(units)>1:
            print(f"Multiple units found for {parameter}: {units}")
    return df


In [ ]:
dc = datacube.Datacube()
measurements = dc.list_measurements().loc['wq_annual']

## TSS

Annual spatial pattern and per station timesries

In [ ]:
parameter = 'TSS'
df=read_insitu(parameter)

In [ ]:
# how to combine chla
chla_msi = ["ndssi_rg_msi_agm"]
chla_oli = ["ndssi_rg_oli_agm"]
chla_tm =["ndssi_rg_tm_agm"]

In [ ]:
years = np.sort(df.datetime.dt.year.unique())
#years = np.array([year for year in years if year<=2012])
years

In [ ]:
# build dataframe with in-situ summary and WQ metrics

mean_dfs = []
for i, year in enumerate(years):
    print(year)
    df_year = df[df["datetime"].dt.year==year]
    mean_df = (
        df_year.groupby(["GEMS Station Number", "Latitude", "Longitude"], as_index=False)
          ["Value"].mean()
    )
    mean_df["year"] = year

    if year <=2012:
        chla_bands = chla_tm
    elif year <2017:
        chla_bands = chla_oli
    else:
        chla_bands = chla_oli + chla_msi

    try:
        ds = dc.load(product='wq_annual', 
            lat = (df_year.Latitude.min()-0.001, df_year.Latitude.max()+0.001),
            lon = (df_year.Longitude.min()-0.001, df_year.Longitude.max()+0.001),
            time=f"{year}",
            measurements= chla_bands,
            resolution = (-0.0005, 0.0005),
            resampling = "nearest",
            output_crs = "EPSG:4326",
            dask_chunks={}).squeeze()

        vals = []
        for _, row in mean_df.iterrows():
            val = ds[chla_bands[0]].sel(latitude=row["Latitude"], longitude=row["Longitude"], method="nearest").values.item()*measurements.loc[chla_bands[0]]['scale_factor'] + measurements.loc[chla_bands[0]]['add_offset']
            n_bands = len(chla_bands)
            if n_bands >1:
                for band in chla_bands[1:]:
                    val += ds[band].sel(latitude=row["Latitude"], longitude=row["Longitude"], method="nearest").values.item()*measurements.loc[band]['scale_factor'] + measurements.loc[band]['add_offset']
                val = val/n_bands
            vals.append(val)
    except:
        vals = []
        
    mean_df["SatValue"] = vals
    mean_dfs.append(mean_df)

annual_df = pd.concat(mean_dfs, ignore_index=True)

In [ ]:
annual_df = pd.concat(mean_dfs, ignore_index=True)

In [ ]:
df_long = annual_df.melt(
    id_vars=["GEMS Station Number", "Latitude", "Longitude", "year"],
    value_vars=["Value", "SatValue"],
    var_name="Type",
    value_name="Measurement"
)

In [ ]:
g = sns.FacetGrid(df_long, col="GEMS Station Number", col_wrap=3, height=4, aspect=1.2)
g.map_dataframe(sns.lineplot, x="year", y="Measurement", hue="Type", marker="o")
g.add_legend()
g.set_titles("{col_name}")
g.set_axis_labels("Year", "Value")
plt.tight_layout()

In [ ]:
# Determine min/max across all measurements for a consistent color scale
vmin = df_long["Measurement"].quantile(0.1)
vmax = df_long["Measurement"].quantile(0.9)

# Create FacetGrid
g = sns.FacetGrid(
    df_long,
    row="year",
    col="Type",
    height=4,
    aspect=1.2,
    margin_titles=True
)

# Plot using matplotlib scatter with fixed vmin/vmax
def scatter_fixed_color(data, color, **kwargs):
    plt.scatter(
        data["Longitude"],
        data["Latitude"],
        c=data["Measurement"],
        cmap="viridis",
        s=80,
        edgecolor="k",
        vmin=vmin,
        vmax=vmax
    )

g.map_dataframe(scatter_fixed_color)

# Add a single colorbar for the whole figure
fig = g.fig
cax = fig.add_axes([1, 0.2, 0.02, 0.6])  # [left, bottom, width, height]
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
fig.colorbar(sm, cax=cax, label="Measurement")

g.set_axis_labels("Longitude", "Latitude")
g.set_titles(row_template="Year: {row_name}", col_template="{col_name}")

In [ ]:
corr = annual_df["Value"].corr(annual_df["SatValue"])
print(f"Pearson correlation (all data): {corr:.3f}")

In [ ]:
station_corrs = annual_df.groupby("GEMS Station Number").apply(
    lambda g: g["Value"].corr(g["SatValue"])
)

print(station_corrs)

In [ ]:
spearman_corr = annual_df["Value"].corr(annual_df["SatValue"], method="spearman")
print(f"Spearman correlation: {spearman_corr:.3f}")

In [ ]:
# Create geometry from lat/lon
geometry = gpd.points_from_xy(annual_df["Longitude"], annual_df["Latitude"])

# Build GeoDataFrame
annual_df = gpd.GeoDataFrame(annual_df, geometry=geometry, crs="EPSG:4326")

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))  # 3 rows, 4 columns
axes = axes.ravel()

for i, year in enumerate(years):
    # Check spatial distribution
    mean_df = annual_df[annual_df.year==year]
    
    sc = axes[i].scatter(
    mean_df["Longitude"],
    mean_df["Latitude"],
    c=mean_df["Value"],
    cmap="viridis",
    s=60,
    edgecolor="k"
)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))  # 3 rows, 4 columns
axes = axes.ravel()

for i, year in enumerate(years):
    # Check spatial distribution
    mean_df = annual_df[annual_df.year==year]

    corr = mean_df["Value"].corr(mean_df["SatValue"])
    print(f"Pearson correlation {year}: {corr:.3f}")

    axes[i].scatter(mean_df["Value"], mean_df["SatValue"], edgecolor="k")
    axes[i].set_xlabel("Station mean value")
    axes[i].set_ylabel("Satellite annual value")
    axes[i].set_title(year)
    lim = [
    min(mean_df["Value"].min(), mean_df["SatValue"].min()),
    max(mean_df["Value"].max(), mean_df["SatValue"].max())
    ]
    axes[i].plot(lim, lim, 'r--', label='1:1 line')

In [ ]:
year = 2005

In [ ]:
df_year = df[df["datetime"].dt.year==year]
mean_df = (
        df_year.groupby(["GEMS Station Number", "Latitude", "Longitude"], as_index=False)
          ["Value"].mean()
    )
    
ds = dc.load(product='wq_annual', 
             lat = (df_year.Latitude.min()-0.001, df_year.Latitude.max()+0.001),
             lon = (df_year.Longitude.min()-0.001, df_year.Longitude.max()+0.001),
             time=f"{year}",
             measurements=chla_bands,
             resolution = (-0.001, 0.001),
             resampling = "nearest",
             output_crs = "EPSG:4326",
             dask_chunks={}).squeeze()

vals, vals_1, vals_2 = [], [], []
for _, row in mean_df.iterrows():
    val = ds[chla_bands[0]].sel(latitude=row["Latitude"], longitude=row["Longitude"], method="nearest").values.item()*measurements.loc[chla_bands[0]]['scale_factor'] + measurements.loc[chla_bands[0]]['add_offset']
    vals_1.append(val)
    for band in chla_bands[1:]:
        val_2 = ds[band].sel(latitude=row["Latitude"], longitude=row["Longitude"], method="nearest").values.item()*measurements.loc[band]['scale_factor'] + measurements.loc[band]['add_offset']
        val +=val_2
        vals_2.append(val_2)
    val = val/len(chla_bands)
    vals.append(val)

mean_df["sat_value"] = vals
mean_df["sat_value_1"] = vals_1
mean_df["sat_value_2"] = vals_2

fig, axes = plt.subplots(2, 2, figsize=(16, 12))  # 3 rows, 4 columns
axes = axes.ravel()

sc = axes[0].scatter(
    mean_df["Longitude"],
    mean_df["Latitude"],
    c=mean_df["Value"],
    cmap="viridis",
    s=60,
    edgecolor="k"
)
axes[0].set_title("In situ")
    
sc = axes[1].scatter(
    mean_df["Longitude"],
    mean_df["Latitude"],
    c=mean_df["sat_value"],
    cmap="viridis",
    s=60,
    edgecolor="k"
)
axes[1].set_title("Satellite mean")

sc = axes[2].scatter(
    mean_df["Longitude"],
    mean_df["Latitude"],
    c=mean_df["sat_value_1"],
    cmap="viridis",
    s=60,
    edgecolor="k"
)
axes[2].set_title(chla_bands[0])

sc = axes[3].scatter(
    mean_df["Longitude"],
    mean_df["Latitude"],
    c=mean_df["sat_value_2"],
    cmap="viridis",
    s=60,
    edgecolor="k"
)
axes[3].set_title(chla_bands[1])

In [ ]:
dc.list_measurements().loc['wq_annual']